In [10]:
import torch 
import torch.nn as nn
import math 

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, d_ff, dropout=0.1, max_length=50_000):
        super().__init__()
        self.d_model = d_model
        self.vocab_emb = nn.Embedding(vocab_size, d_model)
        # a simple linear pos encoding
        # self.pos_encoding = nn.Embedding(max_length, d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=n_head,
            num_encoder_layers=n_layer,
            num_decoder_layers=n_layer,
            dim_feedforward=d_ff,
            dropout=dropout
        )
        self.ow = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('pos_encoding', self.create_sin_cos_position_embedding(max_length, d_model))
        
    def create_sin_cos_position_embedding(self, max_length, d_model):
        position = torch.arange(0, max_length).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10_000.0) / d_model))
        pe = torch.zeros(max_length, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(1)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        src_seq_len = src.size(-2)
        tgt_seq_len = tgt.size(-2)
        src = self.dropout(self.vocab_emb(src) + self.pos_encoding[:src_seq_len, :])
        tgt = self.dropout(self.vocab_emb(tgt) + self.pos_encoding[:tgt_seq_len, :])
        out = self.transformer(src, tgt, src_mask=src_mask, tgt_mask=tgt_mask)
        return self.ow(out)


In [11]:

# Example usage
vocab_size = 10000
embed_size = 512
num_heads = 8
num_encoder_layers = 6
num_decoder_layers = 6
forward_expansion = 2048
dropout = 0.1
max_length = 100
    
model = TransformerModel(
    vocab_size,
    embed_size,
    num_heads,
    num_encoder_layers,
    forward_expansion,
    dropout,
    max_length
)

# Dummy input
src = torch.randint(0, vocab_size, (50, 32))  # (source sequence length, batch size)
trg = torch.randint(0, vocab_size, (50, 32))  # (target sequence length, batch size)

output = model(src, trg)
print(output.shape)  # Expected shape: (target sequence length, batch size, vocab size)


/home/kennethwang/miniconda3/envs/py-notebook/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


torch.Size([50, 32, 10000])


In [13]:
# Load TED talks dataset for Portuguese-English translation
import tensorflow_datasets as tfds
import tensorflow as tf
import torch.optim as optim
import time

# Load the dataset
train_examples, val_examples, test_examples = tfds.load(
    'ted_hrlr_translate/pt_to_en',
    split=['train', 'validation', 'test'],
    as_supervised=True)

# Create tokenizers
tokenizers = {}
for lang, data in [('pt', [ex[0].numpy().decode('utf-8') for ex in train_examples]), 
                   ('en', [ex[1].numpy().decode('utf-8') for ex in train_examples])]:
    tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
        data, target_vocab_size=2**13)
    tokenizers[lang] = tokenizer

# Constants for special tokens
START_TOKEN = [tokenizers['pt'].vocab_size]
END_TOKEN = [tokenizers['pt'].vocab_size + 1]
VOCAB_SIZE_PT = tokenizers['pt'].vocab_size + 2
VOCAB_SIZE_EN = tokenizers['en'].vocab_size + 2

# Preprocessing function
def preprocess_text(pt, en):
    pt = START_TOKEN + tokenizers['pt'].encode(pt.numpy().decode('utf-8')) + END_TOKEN
    en = START_TOKEN + tokenizers['en'].encode(en.numpy().decode('utf-8')) + END_TOKEN
    return pt, en

def tf_preprocess(pt, en):
    return tf.py_function(preprocess_text, [pt, en], [tf.int64, tf.int64])

# Create TF datasets
BUFFER_SIZE = 20000
BATCH_SIZE = 32
MAX_LENGTH = 40

def filter_max_length(pt, en):
    return tf.logical_and(tf.size(pt) <= MAX_LENGTH,
                         tf.size(en) <= MAX_LENGTH)

train_dataset = (train_examples
                .map(tf_preprocess)
                .filter(filter_max_length)
                .shuffle(BUFFER_SIZE)
                .padded_batch(BATCH_SIZE, padded_shapes=([None], [None]))
                .prefetch(tf.data.AUTOTUNE))

val_dataset = (val_examples
              .map(tf_preprocess)
              .filter(filter_max_length)
              .padded_batch(BATCH_SIZE, padded_shapes=([None], [None])))

# Convert TF dataset to PyTorch
def tf_to_torch(tf_dataset):
    for pt_batch, en_batch in tf_dataset:
        pt_tensor = torch.LongTensor(pt_batch.numpy())
        en_tensor = torch.LongTensor(en_batch.numpy())
        yield pt_tensor.T, en_tensor.T  # Transpose to match PyTorch expected shape

# Training function
def train_epoch(model, optimizer, criterion, train_data, device):
    model.train()
    losses = 0
    for src, tgt in tf_to_torch(train_data):
        src = src.to(device)
        tgt = tgt.to(device)
        
        # Create masks
        src_mask = None  # For encoder self-attention
        tgt_len = tgt.shape[0] - 1  # Adjust for teacher forcing
        # Create square subsequent mask for decoder self-attention
        tgt_mask = torch.triu(torch.ones(tgt_len, tgt_len) * float('-inf'), diagonal=1).to(device)
        
        optimizer.zero_grad()
        output = model(src, tgt[:-1], src_mask, tgt_mask)
        
        loss = criterion(output.reshape(-1, output.shape[-1]), tgt[1:].reshape(-1))
        loss.backward()
        optimizer.step()
        
        losses += loss.item()
    
    return losses / len(list(train_data))

# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Update model parameters for new vocabulary sizes
model.vocab_emb = nn.Embedding(VOCAB_SIZE_PT, embed_size)
model.ow = nn.Linear(embed_size, VOCAB_SIZE_EN)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Using 0 as padding index

# Training loop
NUM_EPOCHS = 10
for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    train_loss = train_epoch(model, optimizer, criterion, train_dataset, device)
    end_time = time.time()
    
    print(f"Epoch: {epoch+1}, Train loss: {train_loss:.3f}, Epoch time: {(end_time - start_time):.2f}s")


2025-03-30 17:22:50.834884: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:183: Filling up shuffle buffer (this may take a while): 6162 of 20000
2025-03-30 17:23:10.835576: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:183: Filling up shuffle buffer (this may take a while): 18266 of 20000
2025-03-30 17:23:13.603171: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [1,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [29,0,0] Assertion `t >= 0 && t < n_classes` failed.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
